In [ ]:
import torch
import pandas as pd
from transformers import BertModel, BertTokenizer

# 加载预训练模型和 tokenizer
model_path = '/home/yuantao/code/LLM/biobert-base-cased-v1.2'
model = BertModel.from_pretrained(model_path)
tokenizer = BertTokenizer.from_pretrained(model_path)
model.eval()

# 读取数据
# data = pd.read_csv('/home/yuantao/code/LLM/csv/分类型输出/gemma2/CANCER_go_features3.csv', encoding='utf-8')
data = pd.read_csv('/home/yuantao/code/LLM/csv/分类型输出/gemma2/gemma2_PAN-CANCER_string_new_neiber2.csv', encoding='utf-8')

# 定义需要处理的 statement 列
statement_fields = ['self_statement', 'neighbor_statement', 'together_statement']
weights = torch.tensor([1.0, 0.75, 0.5])  # 你可以根据需要修改权重

def get_bert_embeddings(texts, model, tokenizer):
    """批量获取BERT嵌入（CLS向量）"""
    inputs = tokenizer(texts, return_tensors='pt', padding=True, truncation=True, max_length=1024)
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state[:, 0, :]  # [CLS] token

# 处理每一行数据
results = []
for idx, row in data.iterrows():
    print(f"处理第 {idx} 行")
    
    texts = [row[field] if isinstance(row[field], str) else "" for field in statement_fields]
    embeddings = get_bert_embeddings(texts, model, tokenizer)
    
    # 分别保存三种statement向量
    self_emb = embeddings[0]
    neighbor_emb = embeddings[1]
    together_emb = embeddings[2]

    combined = torch.cat([self_emb, neighbor_emb, together_emb]) 
    results.append(combined)

# 转换为Tensor
final_tensor = torch.stack(results)  # shape: (N, 3072)

# 保存为 .pt 文件
torch.save(final_tensor, '/home/yuantao/code/LLM/csv/分类型输出/gemma2/gemma2_PAN-CANCER_string_new_neiber2.pt')
print(f"保存完成，shape: {final_tensor.shape}")

In [ ]:
import torch
import pandas as pd
from transformers import BertModel, BertTokenizer

# 加载预训练模型和tokenizer
model_path = '/home/yuantao/code/LLM/biobert-base-cased-v1.2'
model = BertModel.from_pretrained(model_path)
tokenizer = BertTokenizer.from_pretrained(model_path)

# 读取数据
data = pd.read_csv('/home/yuantao/code/LLM/csv/分类型输出/gemma2/PAN-CANCER_string_go2.csv', encoding='utf-8')

# 定义特征列
bp_features = ["BP_Feature1", "BP_Feature2", "BP_Feature3"]
mf_features = ["MF_Feature1", "MF_Feature2", "MF_Feature3"]
cc_features = ["CC_Feature1", "CC_Feature2", "CC_Feature3"]

# 设置权重
weights = torch.linspace(1.0, 0.5, steps=3)  # 生成[1.0, 0.75, 0.5]

def get_bert_embeddings(texts, model, tokenizer):
    """批量获取BERT嵌入"""
    inputs = tokenizer(texts, return_tensors='pt', 
                      padding=True, truncation=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state[:, 0, :]  # 取[CLS] token

def weighted_mean(embeddings, weights):
    """带权重的特征平均"""
    weighted = embeddings * weights.view(-1, 1)
    return weighted.sum(dim=0) / weights.sum()

# 处理每一行数据
results = []
for _, row in data.iterrows():
    # 处理BP特征
    print(f"处理行: {_}")
    # bp_texts = [row[f] for f in bp_features]
    bp_texts = [str(row[f]).strip() if pd.notna(row[f]) and str(row[f]).strip() else "none" for f in bp_features]
    bp_embeddings = get_bert_embeddings(bp_texts, model, tokenizer)
    bp_mean = weighted_mean(bp_embeddings, weights)
    
    # 处理MF特征
    # mf_texts = [row[f] for f in mf_features]
    mf_texts = [str(row[f]).strip() if pd.notna(row[f]) and str(row[f]).strip() else "none" for f in mf_features]

    mf_embeddings = get_bert_embeddings(mf_texts, model, tokenizer)
    mf_mean = weighted_mean(mf_embeddings, weights)
    
    # 处理CC特征
    # cc_texts = [row[f] for f in cc_features]
    cc_texts = [str(row[f]).strip() if pd.notna(row[f]) and str(row[f]).strip() else "none" for f in cc_features]
    cc_embeddings = get_bert_embeddings(cc_texts, model, tokenizer)
    cc_mean = weighted_mean(cc_embeddings, weights)
    
    # 计算总平均
    all_mean = (bp_mean + mf_mean + cc_mean) / 3
    
    # 合并结果
    combined = torch.cat([bp_mean, mf_mean, cc_mean, all_mean])
    results.append(combined)

# 转换为最终的torch张量
final_data = torch.stack(results)
print(f"最终数据形状: {final_data.shape}")  # 应为 (样本数, 4 * 768)
import torch_geometric
dataframe = torch_geometric.data.Data()
dataframe['BP'] = final_data[:, 0:768]
dataframe['MF'] = final_data[:, 768:1536]
dataframe['CC'] = final_data[:, 1536:2304]
dataframe['All'] = final_data[:, 2304:3072]
torch.save(dataframe, '/home/yuantao/code/LLM/csv/分类型输出/gemma2/PAN-CANCER_string_go_features_new3.pt')
print(dataframe['All'].size())
